In [1]:
# Library imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter
from pathlib import Path
from collections import Counter
from collections import defaultdict
import os

# Visualization settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# mostra o dataframe para caber certinho na tela
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)

In [2]:
# acessar a pasta experiments
dir_experiments="../experiments/prototypeEvaluation/results"

# Estrutura para armazenar os caminhos dos arquivos 
results_files = defaultdict(dict)

for folder in sorted(os.listdir(dir_experiments)):
    path = os.path.join(dir_experiments, folder)
    for file in sorted(os.listdir(path)):
       if file.endswith("_results.csv") or file.endswith("_layer3_test1-classified-images.cvs"):
                results_files[folder][file] = os.path.join(path, file)
           

In [3]:
results_data = []

for folder, contents in results_files.items():
    superpixel = int(folder.split("_")[0].replace('super', ''))
    nimage = int(folder.split("_")[1].replace('images', ''))
    technique = folder.split('_')[2]
    # print(f"{folder=}")
    # print(f"{superpixel=}, {nimage=}, {technique=}")
    for file, dir in contents.items():
        if isinstance(file, str) and file.endswith('_results.csv'):
            results_file = dir
            seed_value = int(file.replace('_results.csv', '').replace('seed', ''))
            with open(results_file, 'r') as f:
                lines = f.readlines()
                if len(lines) >= 4:
                    # Extract accuracy and kappa metrics from the first two lines
                    class_accuracies = list(map(float, lines[0].strip().split(';')[:9]))
                    class1_accuracy, class2_accuracy = class_accuracies[0], class_accuracies[1]
                    class3_accuracy, class4_accuracy = class_accuracies[2], class_accuracies[3]
                    class5_accuracy, class6_accuracy = class_accuracies[4], class_accuracies[5]
                    class7_accuracy, class8_accuracy = class_accuracies[6], class_accuracies[7]  
                    class9_accuracy = class_accuracies[8]
                    kappa, global_accuracy = map(float, lines[1].strip().split(';')[:2])
                    # Try to extract nfeat from line 3 or 5 if available
                    if len(lines) > 5 and ': ' in lines[5]:
                        try:
                            nfeat = int(lines[5].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    elif len(lines) > 3 and ': ' in lines[3]:
                        try:
                            nfeat = int(lines[3].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    # Append the extracted metrics to the results list
                    results_data.append({
                        'superpixel': superpixel,
                        'nimage': nimage,
                        'technique': technique,
                        'seed': seed_value,
                        'class1_accuracy': class1_accuracy,
                        'class2_accuracy': class2_accuracy,
                        'class3_accuracy': class3_accuracy,
                        'class4_accuracy': class4_accuracy,
                        'class5_accuracy': class5_accuracy,
                        'class6_accuracy': class6_accuracy,
                        'class7_accuracy': class7_accuracy,
                        'class8_accuracy': class8_accuracy,
                        'class9_accuracy': class9_accuracy,
                        'kappa': kappa,
                        'global_accuracy': global_accuracy,
                        'nfeat': nfeat
                    })


# Convert the results list to a DataFrame
df_technique = pd.DataFrame(results_data)

# Save the DataFrame to a CSV file for further analysis
df_technique.to_csv('prototype_results_summary.csv', index=False)


In [4]:
# Define the metrics to be analyzed (excluding 'nfeat' for summary statistics)
metrics = ['class1_accuracy', 'class2_accuracy', 'class3_accuracy', 'class4_accuracy', 'class5_accuracy', 'class6_accuracy', 'class7_accuracy', 'class8_accuracy', 'class9_accuracy', 'kappa', 'global_accuracy', 'nfeat']

# Agrupa por superpixel, nimage e technique e calcula média e desvio padrão para cada métrica
summary_stats = df_technique.groupby(['superpixel', 'nimage', 'technique'])[metrics[:-1]].agg(['mean', 'std']).reset_index()

# Rename columns for easier access (e.g., 'kappa_mean', 'global_accuracy_mean')
summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]

# Arrange the DataFrame according to the order of superpixel values
# superpixels_values = sorted(superpixels_values)
# summary_stats = summary_stats.set_index('superpixel_').loc[superpixels_values].reset_index()

# Select only the metrics of interest for visualization and highlight the highest values
summary_stats_copy = summary_stats[['superpixel_', 'nimage_', 'technique_', 'kappa_mean', 'kappa_std', 'global_accuracy_mean', 'global_accuracy_std']].style.highlight_max(
    subset=['kappa_mean', 'global_accuracy_mean'], color='gray'
)

# Format values as percentages for better presentation
summary_stats_copy = summary_stats_copy.format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

# mostrar apenas cossine
summary_stats[summary_stats['technique_'] == 'cossine']

,superpixel_,nimage_,technique_,class1_accuracy_mean,class1_accuracy_std,class2_accuracy_mean,class2_accuracy_std,class3_accuracy_mean,class3_accuracy_std,class4_accuracy_mean,class4_accuracy_std,class5_accuracy_mean,class5_accuracy_std,class6_accuracy_mean,class6_accuracy_std,class7_accuracy_mean,class7_accuracy_std,class8_accuracy_mean,class8_accuracy_std,class9_accuracy_mean,class9_accuracy_std,kappa_mean,kappa_std,global_accuracy_mean,global_accuracy_std
0,50,2,cossine,0.954023,0.0,0.95,0.0,0.810811,0.0,0.967213,0.0,0.934911,0.0,0.984043,0.0,0.934426,0.0,0.966102,0.0,0.965311,0.0,0.925582,0.0,0.958545,0.0
2,50,3,cossine,0.954023,0.0,0.95,0.0,0.810811,0.0,0.967213,0.0,0.934911,0.0,0.984043,0.0,0.934426,0.0,0.966102,0.0,0.965311,0.0,0.925582,0.0,0.958545,0.0
4,50,4,cossine,0.954023,0.0,0.95,0.0,0.810811,0.0,0.967213,0.0,0.934911,0.0,0.984043,0.0,0.934426,0.0,0.966102,0.0,0.965311,0.0,0.925582,0.0,0.958545,0.0
6,50,5,cossine,0.954023,0.0,0.95,0.0,0.810811,0.0,0.967213,0.0,0.934911,0.0,0.984043,0.0,0.934426,0.0,0.966102,0.0,0.965311,0.0,0.925582,0.0,0.958545,0.0
